## Hybrid Model

In [1]:
import os
import ast
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler

os.chdir(r"C:\Users\Saeed\Desktop\neural_network_project")

MODELS_DIR = "models"
HYBRID_DIR = "models/hybrid"
os.makedirs(HYBRID_DIR, exist_ok=True)

print("Ready!")

Ready!


In [2]:
print("Loading all models...")

# ── Model 1: Content-Based ────────────────────────────────
content_df  = pd.read_csv(f"{MODELS_DIR}/content_based/content_df.csv")
content_sim = np.load(f"{MODELS_DIR}/content_based/cosine_sim.npy")
content_idx = pd.Series(
    content_df.index,
    index=content_df["title"].str.lower()
).drop_duplicates()
print("  [1/5] Content-Based loaded")

# ── Model 3: Metadata ─────────────────────────────────────
meta_df  = pd.read_csv(f"{MODELS_DIR}/metadata/metadata_df.csv")
meta_sim = np.load(f"{MODELS_DIR}/metadata/cosine_sim_meta.npy")
meta_idx = pd.Series(
    meta_df.index,
    index=meta_df["title"].str.lower()
).drop_duplicates()
print("  [2/5] Metadata loaded")

# ── Model 4: Visual ───────────────────────────────────────
visual_df  = pd.read_csv(f"{MODELS_DIR}/visual/visual_df.csv")
visual_sim = np.load(f"{MODELS_DIR}/visual/visual_sim.npy")
visual_idx = pd.Series(
    visual_df.index,
    index=visual_df["title"].str.lower()
).drop_duplicates()
print("  [3/5] Visual loaded")

# ── Model 6: Popularity ───────────────────────────────────
pop_df = pd.read_csv(f"{MODELS_DIR}/popularity/popularity_df.csv")
print("  [4/5] Popularity loaded")

# ── Master DataFrame ──────────────────────────────────────
master = pd.read_csv("data/processed/movies_master.csv")
master_idx = pd.Series(
    master.index,
    index=master["title"].str.lower()
).drop_duplicates()
print("  [5/5] Master loaded")

print(f"\nAll models loaded!")

Loading all models...
  [1/5] Content-Based loaded
  [2/5] Metadata loaded
  [3/5] Visual loaded
  [4/5] Popularity loaded
  [5/5] Master loaded

All models loaded!


In [3]:
def hybrid_recommend(
    title      : str,
    n          : int   = 10,
    w_content  : float = 0.35,
    w_meta     : float = 0.35,
    w_visual   : float = 0.15,
    w_pop      : float = 0.15,
) -> pd.DataFrame:

    title_lower = title.lower().strip()

    def find_title(idx_series, title_lower):
        if title_lower in idx_series:
            return title_lower
        matches = [t for t in idx_series.index if title_lower in t]
        return matches[0] if matches else None

    t_content = find_title(content_idx, title_lower)
    t_meta    = find_title(meta_idx,    title_lower)
    t_visual  = find_title(visual_idx,  title_lower)

    if not t_content:
        print(f"'{title}' not found!")
        return pd.DataFrame()

    scores = {}

    # Model 1: Content-Based
    if t_content:
        idx = content_idx[t_content]
        for i, s in enumerate(content_sim[idx]):
            mid = content_df.iloc[i]["movie_id"]
            scores.setdefault(mid, {})["content"] = float(s)

    # Model 3: Metadata
    if t_meta:
        idx = meta_idx[t_meta]
        for i, s in enumerate(meta_sim[idx]):
            mid = meta_df.iloc[i]["movie_id"]
            scores.setdefault(mid, {})["meta"] = float(s)

    # Model 4: Visual
    if t_visual:
        idx = visual_idx[t_visual]
        for i, s in enumerate(visual_sim[idx]):
            mid = visual_df.iloc[i]["movie_id"]
            scores.setdefault(mid, {})["visual"] = float(s)

    rows = []
    for mid, s in scores.items():
        rows.append({
            "movie_id" : mid,
            "content"  : s.get("content", 0),
            "meta"     : s.get("meta",    0),
            "visual"   : s.get("visual",  0),
        })

    score_df = pd.DataFrame(rows)

    scaler = MinMaxScaler()
    for col in ["content", "meta", "visual"]:
        if score_df[col].max() > 0:
            score_df[col] = scaler.fit_transform(score_df[[col]])

    pop_map = dict(zip(pop_df["movie_id"], pop_df["weighted_score"]))
    pop_min = pop_df["weighted_score"].min()
    pop_max = pop_df["weighted_score"].max()

    score_df["pop"] = score_df["movie_id"].map(pop_map).fillna(pop_min)
    score_df["pop"] = (score_df["pop"] - pop_min) / (pop_max - pop_min + 1e-8)

    score_df["hybrid_score"] = (
        w_content * score_df["content"] +
        w_meta    * score_df["meta"]    +
        w_visual  * score_df["visual"]  +
        w_pop     * score_df["pop"]
    )

    input_movie = master[master["title"].str.lower() == t_content]
    if not input_movie.empty:
        input_id = input_movie.iloc[0]["movie_id"]
        score_df = score_df[score_df["movie_id"] != input_id]

    top = score_df.nlargest(n, "hybrid_score")

    result = top.merge(
        master[["movie_id","title","genres","overview",
                "vote_average","poster_path","director"]],
        on="movie_id",
        how="left"
    )

    return result[[
        "movie_id","title","genres","director",
        "vote_average","hybrid_score",
        "content","meta","visual","pop",
        "poster_path"
    ]].reset_index(drop=True)

print("hybrid_recommend() ready!")

hybrid_recommend() ready!


In [4]:
test_movies = ["Inception", "The Dark Knight", "Toy Story", 
               "The Godfather", "Pulp Fiction"]

print("=== Hybrid Recommendations ===\n")
for movie in test_movies:
    recs = hybrid_recommend(movie, n=5)
    if not recs.empty:
        print(f"[{movie}]")
        for _, row in recs.iterrows():
            print(f"  → {row['title']:40s} "
                  f"hybrid:{row['hybrid_score']:.3f} "
                  f"(C:{row['content']:.2f} "
                  f"M:{row['meta']:.2f} "
                  f"V:{row['visual']:.2f} "
                  f"P:{row['pop']:.2f})")
        print()

=== Hybrid Recommendations ===

[Inception]
  → Interstellar                             hybrid:0.522 (C:0.24 M:0.58 V:0.63 P:0.94)
  → Tenet                                    hybrid:0.479 (C:0.24 M:0.59 V:0.67 P:0.59)
  → The Dark Knight                          hybrid:0.466 (C:0.19 M:0.44 V:0.69 P:0.95)
  → The Prestige                             hybrid:0.456 (C:0.22 M:0.43 V:0.66 P:0.85)
  → Batman Begins                            hybrid:0.424 (C:0.20 M:0.44 V:0.59 P:0.74)

[The Dark Knight]
  → The Dark Knight Rises                    hybrid:0.625 (C:0.41 M:0.74 V:0.72 P:0.76)
  → Batman Begins                            hybrid:0.581 (C:0.35 M:0.69 V:0.70 P:0.74)
  → Inception                                hybrid:0.461 (C:0.19 M:0.44 V:0.69 P:0.92)
  → Tenet                                    hybrid:0.457 (C:0.18 M:0.57 V:0.72 P:0.59)
  → Memento                                  hybrid:0.446 (C:0.16 M:0.47 V:0.65 P:0.84)

[Toy Story]
  → Toy Story 2                             

In [5]:
movie = "Inception"
print(f"=== Weight Comparison for: {movie} ===\n")

configs = {
    "Balanced"         : (0.35, 0.35, 0.15, 0.15),
    "Content Heavy"    : (0.60, 0.20, 0.10, 0.10),
    "Metadata Heavy"   : (0.20, 0.60, 0.10, 0.10),
    "Popularity Heavy" : (0.25, 0.25, 0.10, 0.40),
}

for name, (wc, wm, wv, wp) in configs.items():
    recs = hybrid_recommend(movie, n=3, 
                            w_content=wc, w_meta=wm, 
                            w_visual=wv, w_pop=wp)
    titles = " | ".join(recs["title"].tolist())
    print(f"{name:20s} → {titles}")

=== Weight Comparison for: Inception ===

Balanced             → Interstellar | Tenet | The Dark Knight
Content Heavy        → Interstellar | Tenet | The Prestige
Metadata Heavy       → Interstellar | Tenet | The Dark Knight
Popularity Heavy     → Interstellar | The Dark Knight | The Prestige


In [6]:
# حفظ الأوزان الافتراضية
hybrid_config = {
    "w_content" : 0.35,
    "w_meta"    : 0.35,
    "w_visual"  : 0.15,
    "w_pop"     : 0.15,
}

joblib.dump(hybrid_config, f"{HYBRID_DIR}/hybrid_config.pkl")
master.to_csv(f"{HYBRID_DIR}/master_df.csv", index=False)

print("Hybrid model saved!")
print(f"Config: {hybrid_config}")

Hybrid model saved!
Config: {'w_content': 0.35, 'w_meta': 0.35, 'w_visual': 0.15, 'w_pop': 0.15}
